In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import random
from torch.optim.swa_utils import AveragedModel, SWALR
import time
import torch.optim as optim

In [2]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
import os
from datetime import datetime

In [3]:
torch.backends.cudnn.benchmark = True

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda")

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
train_data = np.load(r"/content/drive/MyDrive/quickdraw_train.npz")
print(train_data.files)

test_data = np.load(r"/content/drive/MyDrive/quickdraw_test.npz")
print(test_data.files)

['x_train', 'y_train', 'class_names']
['test_images']


In [7]:
X = train_data["x_train"]
y = train_data["y_train"]
class_names = train_data["class_names"]

X_test = test_data["test_images"]

print("Train shape:", X.shape)
print("Labels shape:", y.shape)
print("Test shape:", X_test.shape)
print("Number of classes:", len(class_names))

Train shape: (60000, 784)
Labels shape: (60000,)
Test shape: (15000, 784)
Number of classes: 15


In [8]:
print("Unique labels:", np.unique(y))

Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


In [9]:
print(X.shape)

(60000, 784)


In [10]:
print(X.dtype)
print(X.min(), X.max())

uint8
0 255


In [ ]:
class QuickDrawDataset(Dataset):
    def __init__(self, images, labels=None, augment=False):
        self.images = images.astype(np.float32) / 255.0
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def augment_fn(self, x):
      x = x.reshape(28, 28)
      
      if random.random() > 0.5:
          x = torch.flip(x, dims=[1])
      
      if random.random() > 0.5:
          i, j = random.randint(0, 20), random.randint(0, 20)
          h, w = random.randint(4, 8), random.randint(4, 8)
          x[i:i+h, j:j+w] = 0
      x = x.reshape(784)
      noise = torch.randn_like(x) * 0.03
      return torch.clamp(x + noise, 0, 1)

    def __getitem__(self, idx):
        x = torch.tensor(self.images[idx])

        if self.augment:
            x = self.augment_fn(x)

        if self.labels is not None:
            y = torch.tensor(self.labels[idx]).long()
            return x, y

        return x

In [12]:
class ChampionMLP(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes, dropout):
        super().__init__()

        layers = []
        prev = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [13]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        
        mixed_x, y_a, y_b, lam = mixup_data(x, y, alpha=0.3)

        optimizer.zero_grad()
        outputs = model(mixed_x)
        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
        loss.backward()
        optimizer.step()

        
        with torch.no_grad():
            clean_out = model(x)
        preds = torch.argmax(clean_out, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return accuracy_score(all_labels, all_preds)


def validate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            preds = torch.argmax(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

In [15]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:


LOG_FILE = f"train_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def log(msg):
    print(msg)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

log(f"🗂 Log file: {LOG_FILE}")
log(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")



hidden_sizes = [896, 654, 477, 349, 254, 186]
dropout      = 0.22838018994159953
lr           = 0.001733336434313803
wd           = 1.3321907293603369e-05

EPOCHS       = 40
SWA_START    = 25
SWA_LR       = lr * 0.3








log("\n" + "="*60)
log("🔎 Training a single fixed model (no Optuna search)")
log("="*60)
log(f"Architecture: {hidden_sizes}")
log(f"Dropout: {dropout:.6f} | LR: {lr:.12f} | WD: {wd:.12f}")
log(f"SWA_START: {SWA_START} | SWA_LR: {SWA_LR:.6f}")
log(f"EPOCHS: {EPOCHS}")


cv_scores = []
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


os.makedirs("checkpoints", exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    log(f"\n Fold {fold}/3")

    
    train_ds = QuickDrawDataset(X[train_idx], y[train_idx], augment=True)
    val_ds   = QuickDrawDataset(X[val_idx],   y[val_idx])

    
    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=512)

    
    model = ChampionMLP(784, 15, hidden_sizes, dropout).to(device)

    
    param_count = count_parameters(model)
    log(f"Parameters: {param_count:,}")
    if param_count > 3_000_000:
        log("❌ Exceeds 3M parameter limit — Aborting training.")
        break

    
    optimizer     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler     = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    swa_model     = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
    criterion     = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_val   = 0.0
    patience   = 7
    counter    = 0
    swa_active = False

    best_state_dict = None

    for epoch in range(EPOCHS):
        
        train_acc = train_epoch(model, train_loader, optimizer, criterion)

        
        if epoch >= SWA_START:
            swa_model.update_parameters(model)
            swa_scheduler.step()
            swa_active = True
        else:
            scheduler.step()

        
        if swa_active:
            
            update_bn(train_loader, swa_model, device=device)
            val_acc = validate(swa_model, val_loader)
        else:
            val_acc = validate(model, val_loader)

        msg = (f"Epoch {epoch+1:02d}/{EPOCHS} | "
               f"Train Acc: {train_acc:.4f} | "
               f"Val Acc: {val_acc:.4f}"
               f"{' [SWA]' if swa_active else ''}")
        log(msg)

        
        improved = val_acc > best_val
        if improved:
            best_val = val_acc
            
            best_state_dict = (swa_model.module if swa_active else model).state_dict()

            
            ckpt_path = f"checkpoints/fold{fold}_best.pt"
            torch.save({
                "fold": fold,
                "epoch": epoch + 1,
                "val_acc": best_val,
                "arch": hidden_sizes,
                "dropout": dropout,
                "lr": lr,
                "wd": wd,
                "state_dict": best_state_dict
            }, ckpt_path)
            log(f"💾 Saved checkpoint: {ckpt_path}")
            if swa_active:
                counter = 0
        else:
            if swa_active:
                counter += 1
                if counter >= patience:
                    log("⏹ Early stopping triggered")
                    break

    log(f"✅ Fold {fold} Best Val Acc: {best_val:.4f}")
    cv_scores.append(best_val)

🗂 Log file: train_log_20260302_103538.txt
Started at: 2026-03-02 10:35:38

🔎 Training a single fixed model (no Optuna search)
Architecture: [896, 654, 477, 349, 254, 186]
Dropout: 0.228380 | LR: 0.001733336434 | WD: 0.000013321907
SWA_START: 25 | SWA_LR: 0.000520
EPOCHS: 40

📂 Fold 1/3
Parameters: 1,914,022
Epoch 01/40 | Train Acc: 0.5942 | Val Acc: 0.6827
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 02/40 | Train Acc: 0.6804 | Val Acc: 0.7108
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 03/40 | Train Acc: 0.7084 | Val Acc: 0.7256
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 04/40 | Train Acc: 0.7298 | Val Acc: 0.7396
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 05/40 | Train Acc: 0.7473 | Val Acc: 0.7535
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 06/40 | Train Acc: 0.7617 | Val Acc: 0.7707
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 07/40 | Train Acc: 0.7747 | Val Acc: 0.7772
💾 Saved checkpoint: checkpoints/fold1_best.pt
Epoch 08/40 | Train

In [ ]:

if len(cv_scores) > 0:
    mean_score = float(np.mean(cv_scores))
    log("\n🏁 Training Completed (Fixed Model)")
    log(f"Mean CV Accuracy: {mean_score:.4f}")
    log("="*60)
else:
    log("\n⚠️ No folds completed (likely due to param budget abort).")


🏁 Training Completed (Fixed Model)
Mean CV Accuracy: 0.8191


In [ ]:

import os, json, csv, numpy as np, torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def plot_confusion_to_file(cm, classes, out_path, normalize=True, title="Confusion Matrix"):
    if normalize:
        cm = cm.astype(np.float32)
        row_sums = cm.sum(axis=1, keepdims=True) + 1e-12
        cm = cm / row_sums
    plt.figure(figsize=(9,7))
    sns.heatmap(cm, annot=False, cmap="magma", xticklabels=classes, yticklabels=classes)
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def top_misconfusions(cm, k=2):
    cm_copy = cm.copy()
    np.fill_diagonal(cm_copy, 0)
    flat_idx = np.argsort(cm_copy.ravel())[::-1]
    pairs = np.dstack(np.unravel_index(flat_idx, cm_copy.shape))[0]
    out = []
    for (i, j) in pairs:
        if cm_copy[i, j] > 0 and len(out) < k:
            out.append((i, j, cm_copy[i, j]))
        if len(out) == k:
            break
    return out

def save_misclassified_grid(X_val, y_true, y_pred, pair, class_names, out_path, max_per_pair=24):
    ti, pj = pair
    idxs = np.where((y_true == ti) & (y_pred == pj))[0]
    if len(idxs) == 0:
        return []
    n = min(len(idxs), max_per_pair)
    cols = 6
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(cols*2, rows*2))
    for k in range(n):
        img = X_val[idxs[k]].reshape(28, 28)
        plt.subplot(rows, cols, k+1)
        plt.imshow(img, cmap="gray")
        plt.axis("off")
        plt.title(f"T:{class_names[ti]}\nP:{class_names[pj]}", fontsize=8)
    plt.suptitle(f"Misclassified: {class_names[ti]} → {class_names[pj]} (showing {n}/{len(idxs)})", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    return idxs.tolist()

def confidence_diagnostics(y_true, y_pred, probs, pair):
    ti, pj = pair
    mask = (y_true == ti) & (y_pred == pj)
    if mask.sum() == 0:
        return {"count": 0}
    subset = probs[mask]
    chosen = subset.max(axis=1)
    second = np.partition(subset, -2, axis=1)[:, -2]
    margins = chosen - second
    return {
        "count": int(mask.sum()),
        "mean_confidence": float(np.mean(chosen)),
        "median_confidence": float(np.median(chosen)),
        "mean_margin": float(np.mean(margins)),
        "median_margin": float(np.median(margins)),
    }


skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_splits = list(skf.split(X, y))
num_classes = len(class_names)


best_fold, best_acc = None, -1
for f in [1,2,3]:
    path = f"checkpoints/fold{f}_best.pt"
    if os.path.exists(path):
        ckpt = torch.load(path, map_location="cpu")
        acc = ckpt.get("val_acc", 0.0)
        if acc > best_acc:
            best_acc, best_fold = acc, f

assert best_fold is not None, "No checkpoints found in ./checkpoints"


train_idx, val_idx = fold_splits[best_fold-1]
X_val = X[val_idx]
y_val = y[val_idx]
val_ds = QuickDrawDataset(X_val, y_val)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=512, shuffle=False)


def load_fold_model(fold_id, input_size=784, num_classes=num_classes,
                    hidden_sizes=[896, 654, 477, 349, 254, 186], dropout=0.22838018994159953):
    ckpt_path = f"checkpoints/fold{fold_id}_best.pt"
    ckpt = torch.load(ckpt_path, map_location=device)
    model = ChampionMLP(input_size, num_classes, hidden_sizes, dropout).to(device)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.eval()
    return model, float(ckpt.get("val_acc", 0.0))

model, saved_val = load_fold_model(best_fold)


softmax = torch.nn.Softmax(dim=1)
y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = softmax(logits).cpu().numpy()
        y_probs.append(probs)
        y_pred.append(np.argmax(probs, axis=1))
        y_true.append(yb.numpy())
y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
y_probs = np.concatenate(y_probs)


cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))


out_dir = f"confusion_outputs_fold{best_fold}"
os.makedirs(out_dir, exist_ok=True)


cm_png = os.path.join(out_dir, "confusion_matrix.png")
plot_confusion_to_file(cm, class_names, cm_png, normalize=True,
                       title=f"Confusion Matrix (Fold {best_fold}, ValAcc={saved_val:.4f})")


top2 = top_misconfusions(cm, k=2)


all_rows = []
results_lines = []
for (ti, pj, cnt) in top2:
    pair_name = f"{class_names[ti]} -> {class_names[pj]}"
    grid_path = os.path.join(out_dir, f"miscls_true{ti}_{class_names[ti]}_pred{pj}_{class_names[pj]}.png")
    idxs = save_misclassified_grid(X_val, y_true, y_pred, (ti, pj), class_names, grid_path, max_per_pair=24)
    diag = confidence_diagnostics(y_true, y_pred, y_probs, (ti, pj))

    
    results_lines.append(f"PAIR: {pair_name} | count={cnt}")
    results_lines.append(f"  mean_confidence={diag.get('mean_confidence', 0):.4f}, "
                         f"median_confidence={diag.get('median_confidence', 0):.4f}")
    results_lines.append(f"  mean_margin={diag.get('mean_margin', 0):.4f}, "
                         f"median_margin={diag.get('median_margin', 0):.4f}")
    results_lines.append(f"  saved_grid={os.path.basename(grid_path)}")
    results_lines.append("")

    
    for idx in idxs:
        all_rows.append({
            "true_id": ti,
            "true_name": class_names[ti],
            "pred_id": pj,
            "pred_name": class_names[pj],
            "val_index": int(val_idx[idx])  
        })


results_lines.append("Reverse confusion counts for top pairs:")
for (ti, pj, _cnt) in top2:
    rev_cnt = int(cm[pj, ti])
    results_lines.append(f"  {class_names[pj]} -> {class_names[ti]} : {rev_cnt}")
results_lines.append("")


txt_path = os.path.join(out_dir, "results.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"Champion fold: {best_fold} | Saved Val Acc: {saved_val:.4f}\n")
    f.write(f"Confusion matrix (normalized) saved to: {os.path.basename(cm_png)}\n\n")
    f.write("Top-2 confused pairs (true -> pred):\n")
    for (ti, pj, cnt) in top2:
        f.write(f"- {class_names[ti]} -> {class_names[pj]} : {cnt}\n")
    f.write("\nDetails:\n")
    f.write("\n".join(results_lines))


csv_path = os.path.join(out_dir, "misclassified_indices.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["true_id","true_name","pred_id","pred_name","val_index"])
    writer.writeheader()
    writer.writerows(all_rows)

print(f"[Saved] {cm_png}")
print(f"[Saved] {txt_path}")
print(f"[Saved] {csv_path}")

[Saved] confusion_outputs_fold2/confusion_matrix.png
[Saved] confusion_outputs_fold2/results.txt
[Saved] confusion_outputs_fold2/misclassified_indices.csv
